# 09 · Modelos pré-treinados: o ecossistema Hugging Face

**Programação para Aprendizagem de Máquina** · Especialização em Ciência de Dados

Nas aulas 07 e 08 você treinou redes do zero. Em texto e em imagens reais, quase ninguém faz isso: parte-se de um modelo **pré-treinado** em uma quantidade enorme de dados e adapta-se esse modelo ao problema. O Hugging Face é onde esses modelos estão, e o conjunto de bibliotecas que os torna utilizáveis em poucas linhas.

## O que você será capaz de fazer ao final

1. Encontrar um modelo no Hub e ler o que a página dele diz (e o que não diz).
2. Usar um modelo pronto com `pipeline`.
3. Explicar o que um tokenizador faz e o que são *padding*, truncamento e máscara de atenção.
4. Extrair vetores de um modelo e usá-los para medir semelhança e fazer busca semântica.
5. Fazer o ajuste fino de um modelo pré-treinado com o laço de treino que você já conhece.
6. Comparar três formas de usar texto em um classificador, e escolher com critério.

## Como usar este notebook

| Símbolo | Significado |
|---|---|
| 📌 | Lembrete de algo que você já viu e vai precisar agora |
| 🔮 | **Preveja antes de rodar**: responda mentalmente (ou por escrito) antes de executar a próxima célula |
| ✏️ | Exercício. A dificuldade vai de ★ (fixação) a ★★★ (desafio) |
| ✅ | Célula de verificação: confere se a sua resposta faz sentido. Não altere |
| ⚠️ | Armadilha comum |
| ⏳ | Célula que demora mais de 20 segundos |

Nos exercícios, substitua `...` pelo seu código. Se a célula ✅ terminar sem erro, você pode seguir. Se você travar em um exercício, pule: nenhuma célula posterior depende dele.

> 📌 **Use GPU nesta aula:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. O notebook roda em CPU, com amostras reduzidas no ajuste fino, mas os resultados dessa parte deixam de ser representativos.
>
> ⏳ Na primeira execução, os modelos são baixados (cerca de 1,5 GB no total).

In [ ]:
import importlib.util
import subprocess
import sys

faltando = [p for p, modulo in [("transformers", "transformers"), ("sentence-transformers", "sentence_transformers")] if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)

import time
import warnings
from pathlib import Path
from urllib.request import urlretrieve

warnings.filterwarnings("ignore", message="IProgress not found")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sentence_transformers
import torch
import transformers
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

SEMENTE = 42
torch.manual_seed(SEMENTE)
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "· sentence-transformers", sentence_transformers.__version__, "· torch", torch.__version__, "· dispositivo:", dispositivo)

# Os três modelos da aula. Para experimentar outros, basta trocar o nome aqui.
MODELO_SENTIMENTO = "pysentimiento/bertweet-pt-sentiment"                               # classificador de sentimento em português
MODELO_BERT = "neuralmind/bert-base-portuguese-cased"                                   # BERTimbau: BERT treinado em português do Brasil
MODELO_FRASES = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"           # vetores de frases, multilíngue

AZUL, LARANJA, VERDE, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6b6b6b"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlelocation": "left", "axes.titleweight": "bold", "axes.titlesize": 11})

In [ ]:
URL = (
    "https://raw.githubusercontent.com/americanas-tech/b2w-reviews01/"
    "4639429ec698d7821fc99a0bc665fa213d9fcd5a/B2W-Reviews01.csv"
)
ARQUIVO = Path("B2W-Reviews01.csv")
if not ARQUIVO.exists():
    urlretrieve(URL, ARQUIVO)

df = pd.read_csv(ARQUIVO, dtype={"product_id": str}, low_memory=False).dropna(subset=["recommend_to_a_friend", "review_text"])
df["texto"] = df["review_title"].fillna("") + ". " + df["review_text"]
df["nao_recomenda"] = (df["recommend_to_a_friend"] == "No").astype(int)

# Trabalharemos com amostras pequenas: é nesse regime que os modelos pré-treinados mais ajudam.
N_AJUSTE, N_VALID = (4000, 1000) if dispositivo == "cuda" else (300, 200)
amostra = df.sample(N_AJUSTE + N_VALID, random_state=SEMENTE)[["texto", "overall_rating", "nao_recomenda"]].reset_index(drop=True)
ajuste, valid = amostra.iloc[:N_AJUSTE], amostra.iloc[N_AJUSTE:].reset_index(drop=True)
print(f"ajuste: {len(ajuste)} avaliações · validação: {len(valid)} · não recomendam: {ajuste['nao_recomenda'].mean():.1%}")
if dispositivo == "cpu":
    print("⚠️ Sem GPU: amostras reduzidas. Os resultados do ajuste fino (seção 6) não serão representativos.")

## 1. O ecossistema

| Peça | O que é |
|---|---|
| **Hub** (huggingface.co) | repositório público com centenas de milhares de modelos, conjuntos de dados e demonstrações (*Spaces*) |
| **`transformers`** | biblioteca que baixa um modelo do Hub pelo nome e o entrega pronto para uso em PyTorch |
| **`tokenizers`** | conversão rápida de texto em números, do jeito que cada modelo espera |
| **`datasets`** | carregamento e preparo de conjuntos de dados grandes |
| **`sentence-transformers`** | modelos que transformam frases em vetores |

### Como ler a página de um modelo

Antes de usar um modelo, abra a página dele no Hub (o nome do modelo é o endereço: `huggingface.co/neuralmind/bert-base-portuguese-cased`). Procure:

| Item | Por que importa |
|---|---|
| **Tarefa** | o que o modelo faz: classificar texto, preencher lacunas, gerar texto... |
| **Idiomas** | um modelo treinado só em inglês funciona mal em português |
| **Dados de treino** | de onde vêm, de que época, de que domínio (notícias? redes sociais?) |
| **Licença** | se permite uso comercial; há modelos sem licença declarada |
| **Tamanho** | número de parâmetros: define memória, tempo e custo |
| **Avaliação e limitações** | resultados relatados, vieses conhecidos |
| **Quem publicou** | instituição conhecida, grupo de pesquisa, usuário anônimo |

> ⚠️ **Baixar um modelo é executar algo de terceiros.** Prefira modelos de origem conhecida e no formato `safetensors`, que guarda apenas números. Formatos antigos baseados em `pickle` podem conter código.

## 2. `pipeline`: um modelo pronto em três linhas

A função `pipeline` esconde todas as etapas: baixa o modelo, prepara o texto, executa e traduz a saída.

In [ ]:
from transformers import pipeline

classificador = pipeline("text-classification", model=MODELO_SENTIMENTO, device=dispositivo)

frases = [
    "Produto excelente, chegou antes do prazo!",
    "Não recebi o produto e ninguém responde.",
    "É um celular.",
]
for frase, resultado in zip(frases, classificador(frases)):
    print(f"{resultado['label']:<5}{resultado['score']:.3f}   {frase}")

print("\nRótulos que o modelo conhece:", classificador.model.config.id2label)

Este modelo foi treinado por terceiros, com **tuítes** em português, para classificar sentimento em positivo, negativo e neutro. Ele nunca viu as avaliações da B2W. Funciona nelas?

Uma forma de verificar: aplicar a 300 avaliações de validação e cruzar o rótulo do modelo com a resposta que o cliente deu à pergunta "recomendaria?".

In [ ]:
textos_teste = valid["texto"].head(300).tolist()
inicio = time.perf_counter()
saidas = classificador(textos_teste, truncation=True, max_length=128, batch_size=32)      # textos longos são cortados em 128 tokens
print(f"{len(textos_teste)} avaliações em {time.perf_counter() - inicio:.1f} s")

rotulos_previstos = pd.Series([s["label"] for s in saidas], name="rótulo do modelo")
pd.crosstab(rotulos_previstos, valid["nao_recomenda"].head(300).map({0: "recomenda", 1: "não recomenda"}).rename("resposta do cliente"))

**Observe a tabela.** As avaliações de quem não recomenda concentram-se no rótulo negativo? Para onde vão as "neutras"? Um modelo usado assim, sem nenhum exemplo do nosso problema, chama-se uso ***zero-shot***: é o ponto de partida mais barato que existe, e serve de referência para tudo o que vier depois.

### O que o pré-treino ensinou: preenchendo lacunas

O BERT é pré-treinado com uma tarefa simples: esconder palavras de frases e tentar adivinhá-las. Para acertar, o modelo precisa aprender gramática, vocabulário e um bocado sobre o mundo. O BERTimbau fez isso com bilhões de palavras de páginas brasileiras.

In [ ]:
preencher = pipeline("fill-mask", model=MODELO_BERT, device=dispositivo)
mascara = preencher.tokenizer.mask_token                                   # o símbolo de lacuna deste modelo: [MASK]

for frase in [f"O produto chegou {mascara} do prazo.", f"O atendimento foi {mascara} e não resolveu meu problema."]:
    print(frase)
    for sugestao in preencher(frase, top_k=4):
        print(f"    {sugestao['score']:.3f}  {sugestao['token_str']}")

del preencher                                                              # libera a memória

Ninguém ensinou a esse modelo o que é um prazo de entrega. O que ele sabe veio de ler muito texto. É esse conhecimento que vamos reaproveitar.

Outras tarefas disponíveis pelo `pipeline`, todas com a mesma forma de uso:

| Tarefa | Nome no `pipeline` |
|---|---|
| Classificar texto | `"text-classification"` |
| Classificar sem treino, com rótulos dados na hora | `"zero-shot-classification"` |
| Encontrar nomes de pessoas, lugares, organizações | `"token-classification"` |
| Gerar texto | `"text-generation"` |
| Transcrever áudio | `"automatic-speech-recognition"` |
| Classificar imagens | `"image-classification"` |

### ✏️ Exercício 1 (★): do `pipeline` a uma tabela

Escreva `classificar_em_lote(textos)`, que recebe uma lista de textos, aplica o `classificador` (com truncamento em 128 tokens) e devolve um DataFrame com as colunas `texto`, `rotulo` e `confianca`.

In [ ]:
# ✏️ Exercício 1
def classificar_em_lote(textos):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 1 (não altere)
_tabela = classificar_em_lote(["Adorei, recomendo!", "Veio quebrado.", "ok " * 600])
assert list(_tabela.columns) == ["texto", "rotulo", "confianca"] and len(_tabela) == 3, "confira as colunas e o número de linhas"
assert set(_tabela["rotulo"]) <= set(classificador.model.config.id2label.values()), "os rótulos devem ser os do modelo"
assert _tabela["confianca"].between(0, 1).all()
print("Exercício 1 correto (inclusive com um texto muito longo, que exige truncamento).")
_tabela.assign(texto=_tabela["texto"].str[:40])

## 3. Tokenização: do texto aos números

Uma rede neural só opera com números. O **tokenizador** converte o texto em uma sequência de inteiros, e cada modelo tem o seu: usar o tokenizador de um modelo com outro produz lixo.

In [ ]:
from transformers import AutoTokenizer

tokenizador = AutoTokenizer.from_pretrained(MODELO_BERT)
print("tamanho do vocabulário:", tokenizador.vocab_size)

frase = "Não recebi o carregador do notebook, péssimo atendimento."
print(tokenizador.tokenize(frase))

O vocabulário tem alguns milhares de **pedaços de palavras** (*subwords*). Palavras frequentes são um token só; as raras são quebradas em pedaços, e os pedaços que continuam uma palavra começam com `##`. Com isso o modelo nunca encontra uma palavra "desconhecida": no pior caso, ela vira uma sequência de pedaços pequenos.

Chamar o tokenizador devolve o que o modelo recebe:

In [ ]:
codificado = tokenizador(frase)
print("chaves:        ", list(codificado.keys()))
print("input_ids:     ", codificado["input_ids"])
print("de volta:      ", tokenizador.convert_ids_to_tokens(codificado["input_ids"]))

Dois tokens especiais foram acrescentados: `[CLS]` no início e `[SEP]` no fim. O BERT foi treinado com eles e os espera.

### Lotes: *padding*, truncamento e máscara de atenção

Um lote é um tensor retangular, e as frases têm comprimentos diferentes. Duas operações resolvem:

| Operação | O que faz | Argumento |
|---|---|---|
| ***Padding*** | completa as frases curtas com o token `[PAD]` | `padding=True` |
| **Truncamento** | corta as frases que passam do limite | `truncation=True, max_length=...` |

A **máscara de atenção** (`attention_mask`) marca com 1 os tokens de verdade e com 0 o preenchimento, para que o modelo ignore o que foi acrescentado.

In [ ]:
lote = tokenizador(
    ["Ótimo!", "Não recebi o carregador do notebook, péssimo atendimento."],
    padding=True, truncation=True, max_length=32, return_tensors="pt",               # "pt": devolve tensores do PyTorch
)
print("forma:", tuple(lote["input_ids"].shape), "(frases, tokens)")
print(lote["input_ids"])
print(lote["attention_mask"])
print(tokenizador.convert_ids_to_tokens(lote["input_ids"][0]))

> ⚠️ **Truncar é perder informação.** O BERT aceita no máximo 512 tokens, e o custo de processamento cresce com o quadrado do comprimento. O limite escolhido (`max_length`) é uma decisão: baixo demais corta o final das avaliações longas, que muitas vezes é onde está a reclamação.

### ✏️ Exercício 2 (★★): quantos tokens têm as avaliações?

**a)** Escreva `contar_tokens(textos)`, que devolve uma lista com o número de tokens de cada texto, **sem truncar** e contando os tokens especiais.

**b)** Calcule `contagens_ex2` para os textos de `valid` e guarde em `proporcao_cortada` a proporção de avaliações que seriam truncadas com `max_length=128`.

In [ ]:
# ✏️ Exercício 2
def contar_tokens(textos):
    # SEU CÓDIGO AQUI
    ...


# SEU CÓDIGO AQUI
contagens_ex2 = ...
proporcao_cortada = ...

In [ ]:
# ✅ Verificação do exercício 2 (não altere)
assert contar_tokens(["Ótimo!", ""]) == [len(tokenizador("Ótimo!")["input_ids"]), 2], "conte os tokens especiais; um texto vazio tem 2 tokens ([CLS] e [SEP])"
assert len(contagens_ex2) == len(valid) and max(contagens_ex2) > 128, "não trunque ao contar"
assert abs(proporcao_cortada - np.mean(np.array(contagens_ex2) > 128)) < 1e-9

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.hist(np.clip(contagens_ex2, 0, 300), bins=40, color=AZUL)
ax.axvline(128, color=LARANJA, ls="--"); ax.set_xlabel("tokens por avaliação (valores acima de 300 agrupados)"); ax.set_ylabel("avaliações")
ax.set_title(f"{proporcao_cortada:.1%} das avaliações passam de 128 tokens")
print(f"Exercício 2 correto: mediana de {int(np.median(contagens_ex2))} tokens; palavras por avaliação (mediana): {int(valid['texto'].str.split().str.len().median())}.")

## 4. O que sai do modelo

Carregamos agora o corpo do BERTimbau, sem nenhuma camada final de tarefa.

In [ ]:
from transformers import AutoModel

modelo_base = AutoModel.from_pretrained(MODELO_BERT).to(dispositivo).eval()
print(f"{sum(p.numel() for p in modelo_base.parameters()) / 1e6:.0f} milhões de parâmetros")

with torch.no_grad():
    saida = modelo_base(**lote.to(dispositivo))                # ** desempacota input_ids, attention_mask etc. como argumentos

estados = saida.last_hidden_state
print("forma da saída:", tuple(estados.shape), "= (frases, tokens, dimensão)")

(Ao carregar, a biblioteca pode listar pesos que ficaram sem uso: são os da tarefa de preencher lacunas, que o corpo do modelo não precisa. É esperado.)

O modelo devolve **um vetor para cada token** de cada frase. Esses vetores são *contextuais*: o vetor de "banco" em "sentei no banco" é diferente do de "fui ao banco sacar dinheiro", porque cada camada mistura a informação dos tokens vizinhos (o mecanismo de **atenção**).

Para classificar ou comparar frases, precisamos de **um vetor por frase**. Duas formas comuns de resumir:

| Estratégia | Como | Observação |
|---|---|---|
| Vetor do `[CLS]` | o vetor do primeiro token | é o que a camada de classificação do BERT usa |
| **Média** (*mean pooling*) | média dos vetores dos tokens | costuma funcionar melhor para medir semelhança; precisa **ignorar o *padding*** |

### ✏️ Exercício 3 (★★): média com máscara

Escreva `media_com_mascara(estados, mascara)`:

- `estados` tem forma `(frases, tokens, dimensão)` e `mascara`, forma `(frases, tokens)`, com 1 nos tokens reais e 0 no *padding*;
- devolve um tensor `(frases, dimensão)` com a média dos vetores **apenas dos tokens reais** de cada frase.

Dica: multiplique os estados pela máscara (com um eixo a mais, via `unsqueeze(-1)`), some no eixo dos tokens e divida pelo número de tokens reais.

In [ ]:
# ✏️ Exercício 3
def media_com_mascara(estados, mascara):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 3 (não altere)
_estados = torch.tensor([[[1.0, 2.0], [3.0, 4.0], [100.0, 100.0]],
                         [[1.0, 1.0], [2.0, 2.0], [3.0, 3.0]]])              # 2 frases, 3 tokens, dimensão 2
_mascara = torch.tensor([[1, 1, 0], [1, 1, 1]])                              # o terceiro token da primeira frase é padding
_media = media_com_mascara(_estados, _mascara)
assert tuple(_media.shape) == (2, 2), "a saída deve ter forma (frases, dimensão)"
assert torch.allclose(_media, torch.tensor([[2.0, 3.0], [2.0, 2.0]])), "o padding entrou na média, ou a divisão usou o número errado de tokens"
_reais = media_com_mascara(estados, lote["attention_mask"])
assert tuple(_reais.shape) == (2, estados.shape[-1])
print("Exercício 3 correto. Vetores das duas frases do lote:", tuple(_reais.shape))

In [ ]:
del modelo_base                      # libera a memória antes de carregar o próximo modelo
if dispositivo == "cuda":
    torch.cuda.empty_cache()

## 5. Vetores de frases e busca semântica

O BERT original não foi treinado para que frases parecidas tenham vetores parecidos. Os modelos da biblioteca `sentence-transformers` foram: tomam um modelo como o BERT, aplicam a média com máscara que você acabou de escrever e são ajustados com milhões de pares de frases. O resultado é um ***embedding***: um vetor que representa o **significado** da frase.

In [ ]:
from sentence_transformers import SentenceTransformer, util

modelo_frases = SentenceTransformer(MODELO_FRASES, device=dispositivo)

frases = [
    "O produto chegou com atraso.",
    "A entrega demorou muito mais que o prometido.",
    "The delivery was late.",
    "A bateria do celular dura pouco.",
    "O aparelho descarrega rápido demais.",
    "Adorei a cor do vestido.",
]
vetores = modelo_frases.encode(frases, normalize_embeddings=True)          # normalizados: o produto interno é o cosseno
print("forma:", vetores.shape, "= (frases, dimensão)")

semelhanca = vetores @ vetores.T
fig, ax = plt.subplots(figsize=(6.4, 5))
ax.imshow(semelhanca, cmap="Blues", vmin=0, vmax=1)
for i in range(len(frases)):
    for j in range(len(frases)):
        ax.text(j, i, f"{semelhanca[i, j]:.2f}", ha="center", va="center", fontsize=8, color="white" if semelhanca[i, j] > 0.6 else "black")
ax.set_xticks(range(len(frases)), range(1, len(frases) + 1)); ax.set_yticks(range(len(frases)), [f"{i + 1}. {f[:38]}" for i, f in enumerate(frases)])
ax.spines[:].set_visible(False); ax.set_title("Semelhança de cosseno entre as frases")
plt.tight_layout()

**Observe a matriz.** As frases 1 e 2 dizem a mesma coisa com palavras quase todas diferentes; a 3 diz o mesmo em inglês; a 4 e a 5 tratam de bateria sem compartilhar essa palavra. Compare as semelhanças dentro desses grupos com as semelhanças entre grupos. O TF-IDF das aulas anteriores daria semelhança próxima de zero a frases sem palavras em comum.

### Busca semântica

Com os vetores de um acervo de textos, procurar por **significado** vira uma conta de álgebra linear: o vetor da pergunta contra a matriz do acervo.

In [ ]:
acervo = ajuste["texto"].head(2000).tolist()

inicio = time.perf_counter()
vetores_acervo = modelo_frases.encode(acervo, normalize_embeddings=True, batch_size=64, show_progress_bar=False)
print(f"{len(acervo)} avaliações convertidas em vetores {vetores_acervo.shape} em {time.perf_counter() - inicio:.1f} s")

consulta = "demorou demais para chegar"
vetor_consulta = modelo_frases.encode([consulta], normalize_embeddings=True)
for achado in util.semantic_search(vetor_consulta, vetores_acervo, top_k=5)[0]:           # busca pronta da biblioteca
    print(f"{achado['score']:.3f}  {acervo[achado['corpus_id']][:110]}")

print(f"\nAvaliações do acervo que contêm literalmente a palavra 'demorou': {sum('demorou' in texto.lower() for texto in acervo)}")

Leia os resultados: quantos tratam de atraso na entrega **sem** usar a palavra "demorou"? Troque a consulta e execute de novo. Esta é a peça central dos sistemas de busca modernos e do RAG, que veremos na aula 10.

### ✏️ Exercício 4 (★★): a busca, por dentro

Escreva `mais_parecidos(vetor, matriz, k)`: recebe um vetor de forma `(dimensão,)` e uma matriz `(textos, dimensão)`, ambos **normalizados**, e devolve os **índices** das `k` linhas da matriz mais parecidas com o vetor, da mais para a menos parecida. Use só o NumPy.

In [ ]:
# ✏️ Exercício 4
def mais_parecidos(vetor, matriz, k=5):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 4 (não altere)
_matriz = np.array([[1.0, 0.0], [0.0, 1.0], [0.6, 0.8], [-1.0, 0.0], [0.8, 0.6]])
assert list(mais_parecidos(np.array([1.0, 0.0]), _matriz, k=3)) == [0, 4, 2], "os índices devem vir do mais para o menos parecido"
assert list(mais_parecidos(np.array([0.0, 1.0]), _matriz, k=2)) == [1, 2]
_meus = list(mais_parecidos(vetor_consulta[0], vetores_acervo, k=5))
_da_biblioteca = [a["corpus_id"] for a in util.semantic_search(vetor_consulta, vetores_acervo, top_k=5)[0]]
assert _meus == _da_biblioteca, "o resultado difere do da biblioteca"
print("Exercício 4 correto: a sua busca devolve os mesmos resultados da biblioteca.")

### Vetores como colunas de um classificador

Um *embedding* é uma linha de números: pode alimentar qualquer modelo do scikit-learn. Comparamos, **com os mesmos exemplos de ajuste**, uma regressão logística sobre os vetores e a nossa referência das aulas anteriores, a regressão logística sobre TF-IDF.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

resultados = {}

tfidf = make_pipeline(TfidfVectorizer(min_df=2), LogisticRegression(max_iter=1000)).fit(ajuste["texto"], ajuste["nao_recomenda"])
resultados["TF-IDF + regressão logística"] = roc_auc_score(valid["nao_recomenda"], tfidf.predict_proba(valid["texto"])[:, 1])

X_ajuste = modelo_frases.encode(ajuste["texto"].tolist(), normalize_embeddings=True, batch_size=64, show_progress_bar=False)
X_valid = modelo_frases.encode(valid["texto"].tolist(), normalize_embeddings=True, batch_size=64, show_progress_bar=False)
sobre_vetores = LogisticRegression(max_iter=1000).fit(X_ajuste, ajuste["nao_recomenda"])
resultados["vetores de frases + regressão logística"] = roc_auc_score(valid["nao_recomenda"], sobre_vetores.predict_proba(X_valid)[:, 1])

for nome, valor in resultados.items():
    print(f"{nome:<45}AUC {valor:.4f}   ({len(ajuste)} exemplos de ajuste)")

Nos vetores, o modelo de frases ficou **congelado**: nenhum parâmetro dele foi alterado, e só a regressão logística aprendeu. Anote os dois valores. A seguir, o passo mais forte: alterar o próprio modelo pré-treinado.

In [ ]:
del modelo_frases
if dispositivo == "cuda":
    torch.cuda.empty_cache()

## 6. Ajuste fino

No **ajuste fino** (*fine-tuning*), acrescenta-se ao corpo pré-treinado uma camada de classificação nova e treina-se **tudo**, com o laço de sempre e duas diferenças importantes:

- a **taxa de aprendizado é muito pequena** (da ordem de 0,00002): o modelo já sabe muito, e passos grandes destruiriam esse conhecimento;
- bastam **poucas épocas** (de uma a três).

`AutoModelForSequenceClassification` monta o modelo: corpo do BERTimbau, mais uma camada linear com `num_labels` saídas.

In [ ]:
from transformers import AutoModelForSequenceClassification

torch.manual_seed(SEMENTE)
modelo_ajustado = AutoModelForSequenceClassification.from_pretrained(MODELO_BERT, num_labels=2).to(dispositivo)

A biblioteca avisa, ao carregar, que alguns pesos foram **criados agora** (os da camada de classificação) e que outros foram descartados (os da tarefa de preencher lacunas). É o esperado: a camada nova começa aleatória, e é por isso que o modelo precisa ser treinado antes de servir para alguma coisa.

Os dados entram como na aula 08: um `Dataset` e um `DataLoader`. A novidade é a função que monta o lote (`collate_fn`): ela tokeniza os textos **do lote** e aplica o *padding* só até o comprimento do maior deles, o que economiza muito tempo.

In [ ]:
class Avaliacoes(Dataset):
    def __init__(self, textos, rotulos):
        self.textos, self.rotulos = list(textos), list(rotulos)

    def __len__(self):
        return len(self.rotulos)

    def __getitem__(self, i):
        return self.textos[i], self.rotulos[i]


def montar_lote(itens):
    textos, rotulos = zip(*itens)
    lote = tokenizador(list(textos), padding=True, truncation=True, max_length=128, return_tensors="pt")
    lote["labels"] = torch.tensor(rotulos)                     # o nome "labels" é o que o modelo espera
    return lote


carregador_ajuste = DataLoader(Avaliacoes(ajuste["texto"], ajuste["nao_recomenda"]), batch_size=16, shuffle=True, collate_fn=montar_lote)
carregador_valid = DataLoader(Avaliacoes(valid["texto"], valid["nao_recomenda"]), batch_size=64, collate_fn=montar_lote)

exemplo_de_lote = next(iter(carregador_ajuste))
print({nome: tuple(tensor.shape) for nome, tensor in exemplo_de_lote.items()})

In [ ]:
@torch.no_grad()
def probabilidades(modelo, carregador):
    """Probabilidade da classe 1 para cada exemplo do carregador."""
    modelo.eval()
    partes = []
    for lote in carregador:
        logits = modelo(**lote.to(dispositivo)).logits
        partes.append(torch.softmax(logits, dim=1)[:, 1].cpu())
    return torch.cat(partes).numpy()


auc_antes = roc_auc_score(valid["nao_recomenda"], probabilidades(modelo_ajustado, carregador_valid))
print(f"Antes do ajuste fino: AUC {auc_antes:.4f} (a camada de classificação é aleatória)")

⏳ O treino abaixo percorre os exemplos de ajuste uma vez. Em GPU, deve levar poucos minutos; acompanhe a perda.

In [ ]:
EPOCAS = 1
otimizador = torch.optim.AdamW(modelo_ajustado.parameters(), lr=2e-5, weight_decay=0.01)

inicio = time.perf_counter()
for epoca in range(1, EPOCAS + 1):
    modelo_ajustado.train()
    soma, vistos = 0.0, 0
    for passo, lote in enumerate(carregador_ajuste, start=1):
        lote = lote.to(dispositivo)

        otimizador.zero_grad()                      # 1. zerar
        saida = modelo_ajustado(**lote)             # 2. prever (com "labels" no lote, o modelo já calcula a perda)
        saida.loss.backward()                       # 3 e 4. perda e retropropagação
        otimizador.step()                           # 5. atualizar

        soma += saida.loss.item() * len(lote["labels"]); vistos += len(lote["labels"])
        if passo % 50 == 0 or passo == len(carregador_ajuste):
            print(f"época {epoca} · passo {passo:>3}/{len(carregador_ajuste)} · perda média {soma / vistos:.4f} · {time.perf_counter() - inicio:.0f} s")

resultados["BERTimbau com ajuste fino"] = roc_auc_score(valid["nao_recomenda"], probabilidades(modelo_ajustado, carregador_valid))
print(f"\nDepois do ajuste fino: AUC {resultados['BERTimbau com ajuste fino']:.4f}")

In [ ]:
comparacao = pd.Series(resultados).sort_values()

fig, ax = plt.subplots(figsize=(7.5, 2.2))
ax.hlines(comparacao.index, 0.5, comparacao.values, color=AZUL, lw=1, alpha=0.5)
ax.scatter(comparacao.values, comparacao.index, color=AZUL, s=50, zorder=3)
for nome, valor in comparacao.items():
    ax.text(valor + 0.008, nome, f"{valor:.3f}", va="center", fontsize=9)
ax.set_xlim(0.5, 1.04); ax.set_xlabel("ROC AUC na validação")
ax.set_title(f"Três formas de usar o texto, com {len(ajuste)} exemplos de ajuste")
plt.tight_layout()

**Leia o gráfico com estas perguntas:**

- Qual das três abordagens ficou na frente? Por quanto?
- Na aula 05, o TF-IDF com **todos** os dados (mais de 100 mil avaliações) chegou a cerca de 0,96. Com os poucos milhares de exemplos desta aula, quanto ele alcança? E os modelos pré-treinados?
- Quanto custou cada uma, em tempo de treino e em equipamento?

O conhecimento que o modelo traz do pré-treino substitui, em parte, os exemplos rotulados que você não tem. Essa é a razão prática de usar modelos pré-treinados: **rótulos são caros**.

### ✏️ Exercício 5 (★★): usar o modelo ajustado

Escreva `prever_textos(textos)`, que recebe uma lista de textos quaisquer e devolve um array NumPy com a probabilidade de "não recomenda" para cada um, usando `tokenizador` e `modelo_ajustado`. Cuide do modo de avaliação, dos gradientes e do dispositivo.

In [ ]:
# ✏️ Exercício 5
def prever_textos(textos):
    # SEU CÓDIGO AQUI
    ...

In [ ]:
# ✅ Verificação do exercício 5 (não altere)
_novas = ["Produto excelente, recomendo a todos!", "Não recebi o produto, péssimo atendimento. Não comprem.", "ok " * 600]
_p = prever_textos(_novas)
assert isinstance(_p, np.ndarray) and _p.shape == (3,) and ((_p >= 0) & (_p <= 1)).all(), "devolva um array com uma probabilidade por texto"
assert not modelo_ajustado.training, "coloque o modelo em modo de avaliação"
assert np.allclose(_p[:2], prever_textos(_novas[:2]), atol=1e-4), "a previsão de um texto não pode depender dos outros textos do lote"
assert np.allclose(prever_textos(valid["texto"].head(8).tolist()), probabilidades(modelo_ajustado, carregador_valid)[:8], atol=1e-3), "o resultado difere do obtido pela função de avaliação"
print("Exercício 5 correto. Probabilidade de 'não recomenda':")
for texto, p in zip(_novas, _p):
    print(f"   {p:.3f}  {texto[:60]}")

## 7. Guardar e compartilhar

Um modelo ajustado é guardado com `save_pretrained`, que grava os pesos e a configuração em uma pasta. Antes, vale registrar o **nome das classes** na configuração, para que quem usar o modelo não receba `LABEL_0` e `LABEL_1`.

In [ ]:
modelo_ajustado.config.id2label = {0: "recomenda", 1: "não recomenda"}
modelo_ajustado.config.label2id = {"recomenda": 0, "não recomenda": 1}

modelo_ajustado.save_pretrained("bert-avaliacoes")
tokenizador.save_pretrained("bert-avaliacoes")                     # modelo e tokenizador andam juntos
print(sorted(p.name for p in Path("bert-avaliacoes").iterdir()))

recarregado = pipeline("text-classification", model="bert-avaliacoes", device=dispositivo)      # uma pasta local serve no lugar do nome
recarregado(["Chegou tudo certo, adorei.", "Veio com defeito e não consigo trocar."])

Para publicar no Hub, a pasta é enviada com `modelo.push_to_hub("seu-usuario/nome-do-modelo")`, depois de autenticar com um *token* de escrita guardado nos *Secrets* do Colab. Um modelo publicado deve vir com a sua própria página: dados usados, resultado obtido, limites conhecidos. Você leu páginas assim no começo da aula; é a sua vez de escrever uma (lista de casa).

> ⚠️ **No Colab, a pasta some com a sessão.** Copie para o Drive ou publique no Hub.

## 8. Para levar

### Qual abordagem usar?

| Situação | Ponto de partida razoável |
|---|---|
| Nenhum exemplo rotulado | `pipeline` com modelo pronto (*zero-shot*), ou um LLM (aula 10) |
| Dezenas ou centenas de exemplos | vetores de frases + regressão logística |
| Milhares de exemplos e GPU disponível | ajuste fino |
| Dezenas de milhares de exemplos, textos simples, custo baixo de operação | TF-IDF + modelo linear continua competitivo |
| Buscar textos parecidos, agrupar, eliminar duplicatas | vetores de frases |
| Textos longos (contratos, laudos) | dividir em trechos; ou modelos de contexto longo |

### Cuidados

| Cuidado | Por quê |
|---|---|
| **Licença** | nem todo modelo do Hub pode ser usado comercialmente |
| **Viés** | o modelo reflete os textos em que foi treinado, com seus estereótipos |
| **Domínio** | um modelo de tuítes pode falhar em laudos médicos |
| **Custo de operação** | um BERT é milhares de vezes mais caro por previsão que uma regressão logística |
| **Dados sensíveis** | ao usar um modelo local, os textos não saem da sua máquina; em serviços de terceiros, saem |
| **Versões** | fixe a versão do modelo (`revision=`) e das bibliotecas |

### Erros comuns

| Erro | Consequência | Como evitar |
|---|---|---|
| Tokenizador de um modelo com os pesos de outro | resultados sem sentido, sem mensagem de erro | carregar os dois pelo mesmo nome |
| Esquecer `truncation=True` | erro com textos longos | truncar sempre, com `max_length` escolhido |
| Média dos vetores incluindo o *padding* | vetores piores para frases curtas em lotes longos | média com máscara |
| Taxa de aprendizado "normal" (0,001) no ajuste fino | o modelo desaprende o pré-treino | 0,00001 a 0,00005 |
| Avaliar sem `eval()` | *dropout* ligado, previsões instáveis | `eval()` e `no_grad` |
| Ignorar o aviso de pesos recém-criados e usar o modelo sem treinar | previsões aleatórias | a camada nova precisa de ajuste fino |
| Vários modelos grandes na GPU ao mesmo tempo | falta de memória | apagar (`del`) e `torch.cuda.empty_cache()` |
| Guardar o modelo sem o tokenizador | o modelo fica inutilizável | `save_pretrained` nos dois, na mesma pasta |

### Resumo

- O **Hub** reúne modelos pré-treinados; a página do modelo diz tarefa, idioma, dados, licença e limites.
- `pipeline` usa um modelo pronto em três linhas; é a referência *zero-shot*.
- O **tokenizador** quebra o texto em pedaços de palavras; *padding*, truncamento e máscara de atenção permitem montar lotes.
- O modelo devolve um vetor por token; a **média com máscara** resume a frase em um vetor.
- **Vetores de frases** aproximam textos de mesmo significado: servem para busca semântica e como colunas de qualquer classificador.
- O **ajuste fino** treina o modelo inteiro com taxa de aprendizado pequena e poucas épocas, com o laço da aula 08.
- Modelos pré-treinados rendem mais quando há **poucos exemplos rotulados**.

### O que vem a seguir

O notebook 10 trata dos modelos de linguagem que **geram** texto (LLMs), usados como um componente dentro de um programa: instruções, saída estruturada, avaliação e busca com geração (RAG).

### Para casa

A lista de exercícios está em `09_casa.ipynb`.